# LLM Feedback on Novice Programs: Code-Only (A) vs Verification-Grounded (B)
COMP 5112 Week 2 implementation

### Config

In [ ]:
import os, sys, re, json, ast, time, math, random, sqlite3, pathlib, tempfile, subprocess, hashlib, glob
from concurrent.futures import ThreadPoolExecutor, as_completed

DATA_SOURCE = "intro_prog"
HF_REPO = "koutch/falcon_code"
FC_SUBSET = "submitted"
FC_SPLIT = "test"
CSV_PATH = None
TESTS_JSON = None
TYPE_FILTER = None
IP_REPO = "koutch/intro_prog"
IP_SOURCE = "singapore"
IP_SPLIT = "train"

N_PROBLEMS = 5
PER_PROBLEM = 10
FRAC_CORRECT = 0.2
MAX_LINES = 120
SEED = 42

LLM_PROVIDER = "openai"
OPENAI_MODEL = "gpt-4o"
ANTHROPIC_MODEL = "claude-sonnet-5-5"
TEMPERATURE = 0.0
K_SAMPLES = 3
MAX_WORKERS = 4

TIMEOUT_S = 5
MEM_MB = 512
INCLUDE_SCORE_IN_SIGNAL = DATA_SOURCE != "intro_prog"

CODER2_FRACTION = 0.25
ANNOT_CSV = r"C:\kaggle\working\annotation_sheet.csv"

OUTPUT_DIR = None

def pick_workdir():
    here = os.path.dirname(os.path.abspath(globals().get("__file__", ""))) if "__file__" in globals() else os.getcwd()
    options = [OUTPUT_DIR, "/kaggle/working" if os.path.isdir("/kaggle/working") else None, here, os.getcwd(),
               os.path.join(os.path.expanduser("~"), "falconcode_llm_feedback")]
    for d in options:
        if not d:
            continue
        try:
            os.makedirs(d, exist_ok=True)
            probe = os.path.join(d, ".write_test")
            with open(probe, "w") as f:
                f.write("ok")
            os.remove(probe)
            return os.path.abspath(d)
        except OSError as e:
            print(f"Skipping {d!r}: {e}")
    raise RuntimeError("No writable folder found. Set OUTPUT_DIR to a folder you can write to.")

WORKDIR = pick_workdir()
DB_PATH = os.path.join(WORKDIR, "feedback.db")
print("Output folder:", WORKDIR)
random.seed(SEED)

Output folder: c:\kaggle\working


### Helpers

In [2]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def get_secret(name, ask=False):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    value = os.environ.get(name)
    if not value and ask:
        import getpass
        try:
            value = getpass.getpass(f"Enter {name} (hidden, kept only for this session): ").strip()
        except (EOFError, OSError):
            value = ""
        if value:
            os.environ[name] = value
    return value

def ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

### API key check

In [3]:
KEY_NAME = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}.get(LLM_PROVIDER)
if KEY_NAME:
    if not get_secret(KEY_NAME, ask=True):
        raise RuntimeError(f"No {KEY_NAME} provided. Put it in a .env file next to the notebook, set it as an environment variable, "
                           "or add it as a Kaggle secret. Or set LLM_PROVIDER = 'mock' for a free test.")
    print(f"{KEY_NAME} found for provider {LLM_PROVIDER!r}")

OPENAI_API_KEY found for provider 'openai'


### Setup

In [4]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

### Demo data

In [5]:
DEMO_PROBLEMS = {
    "d_prime": dict(
        prompt="Read an integer n and print True if n is prime, otherwise print False.",
        good="n = int(input())\nif n < 2:\n    print(False)\nelse:\n    print(all(n % i for i in range(2, int(n**0.5) + 1)))\n",
        bug="n = int(input())\nprint(all(n % i for i in range(2, n)))\n",
        tests=[("2", "True"), ("9", "False"), ("1", "False"), ("13", "True")]),
    "d_sum_even": dict(
        prompt="Read a number n and print the sum of the even numbers from 1 to n inclusive.",
        good="n = int(input())\nprint(sum(i for i in range(1, n + 1) if i % 2 == 0))\n",
        bug="n = int(input())\nprint(sum(i for i in range(1, n) if i % 2 == 0))\n",
        tests=[("10", "30"), ("7", "12"), ("2", "2")]),
    "d_reverse": dict(
        prompt="Read a word and print it reversed.",
        good="w = input()\nprint(w[::-1])\n",
        bug="w = input()\nprint(w[1:][::-1])\n",
        tests=[("abc", "cba"), ("a", "a")]),
}

def make_demo():
    rows, tests = [], {}
    for pid, d in DEMO_PROBLEMS.items():
        tests[pid] = [dict(name=f"test_{i+1}", stdin=s, expected=e) for i, (s, e) in enumerate(d["tests"])]
        for i in range(30):
            kind = ["good", "bug", "syntax", "runtime", "bug", "good"][i % 6]
            if kind == "good":
                code, score = d["good"], 1.0
            elif kind == "bug":
                code, score = d["bug"], 0.67
            elif kind == "syntax":
                code, score = d["good"].replace("input()", "input(", 1), 0.0
            else:
                code, score = "x = int(input())\nprint(x / 0)\n", 0.0
            rows.append(dict(problem_id=pid, source_code=f"# attempt {i}\n" + code, score=score, prompt=d["prompt"], type="lab"))
    return pd.DataFrame(rows), tests

### Load FalconCode

In [6]:
NEEDED = ["problem_id", "source_code", "score", "prompt"]

def load_falconcode_hf():
    ensure("datasets"); ensure("huggingface_hub")
    from datasets import load_dataset
    from huggingface_hub import login
    from huggingface_hub import get_token
    token = get_secret("HF_TOKEN") or get_token()
    if not token:
        raise RuntimeError("No Hugging Face token found. On Kaggle add an HF_TOKEN secret; locally set the HF_TOKEN "
                           "environment variable or run `huggingface-cli login` once.")
    login(token=token)
    try:
        ds = load_dataset(HF_REPO, FC_SUBSET, split=FC_SPLIT)
    except Exception as e:
        raise RuntimeError(
            f"Could not load {HF_REPO} ({FC_SUBSET}/{FC_SPLIT}): {e}\n"
            "Check that you accepted the conditions at https://huggingface.co/datasets/koutch/falcon_code "
            "and that Internet is On.") from e
    keep = [c for c in NEEDED + ["type"] if c in ds.column_names]
    return ds.select_columns(keep).to_pandas()

def load_falconcode_csv():
    path = CSV_PATH or next(iter(sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))), None)
    if not path:
        raise FileNotFoundError("No CSV found. Set CSV_PATH to your FalconCode file (on Kaggle you can also add it as a Dataset input).")
    print("reading", path)
    return pd.read_csv(path)

def split_asserts(test_src):
    try:
        body = ast.parse(test_src).body
    except SyntaxError:
        return None
    exprs = []
    for node in body:
        if not isinstance(node, ast.Assert):
            return None
        v = node.test
        exprs += v.values if isinstance(v, ast.BoolOp) and isinstance(v.op, ast.And) else [v]
    return exprs or None

def expr_test(e):
    d = dict(name=ast.unparse(e)[:90], expr=ast.unparse(e))
    if isinstance(e, ast.Compare) and len(e.ops) == 1 and isinstance(e.ops[0], ast.Eq):
        d.update(left=ast.unparse(e.left), right=ast.unparse(e.comparators[0]), name=ast.unparse(e.left)[:90])
    return d

def load_intro_prog():
    ensure("huggingface_hub")
    from huggingface_hub import hf_hub_download

    def fetch(name):
        return hf_hub_download(IP_REPO, f"data/{name}.jsonl", repo_type="dataset")

    try:
        df = pd.read_json(fetch(f"{IP_SOURCE}_data_{IP_SPLIT}"), lines=True, dtype=False)
    except Exception as e:
        raise RuntimeError(f"Could not download {IP_REPO} data/{IP_SOURCE}_data_{IP_SPLIT}.jsonl: {e}\n"
                           "Check that Internet is On. Valid choices: singapore/train, dublin/train, dublin/test, newcaledonia/train.") from e
    df = df.rename(columns={"assingment_id": "assignment_id", "Description": "description"})
    need = ["func_code", "assignment_id", "func_name", "test", "correct"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise KeyError(f"intro_prog is missing columns {missing}. Found: {list(df.columns)}")
    if "description" not in df.columns:
        df["description"] = ""
    try:
        meta = pd.read_json(fetch(f"{IP_SOURCE}_metadata_{IP_SPLIT if IP_SPLIT != 'test' else 'train'}"), lines=True, dtype=False)
        meta = meta.rename(columns={"assingment_id": "assignment_id", "Description": "description"})
        long = {str(r.assignment_id): str(r.description) for r in meta.itertuples() if str(getattr(r, "description", "")).strip()}
    except Exception:
        long = {}
    df["assignment_id"] = df["assignment_id"].astype(str)
    df["description"] = [long.get(a, d) or "" for a, d in zip(df.assignment_id, df.description)]
    df["problem_id"] = IP_SOURCE + "_" + df["assignment_id"]
    df["prompt"] = [f"{d}\nWrite the Python function `{f}`." for d, f in zip(df.description, df.func_name)]
    df["source_code"] = df["func_code"]
    df["score"] = df["correct"].astype(str).str.lower().isin(["true", "1"]).astype(float)
    tests = {}
    for pid, g in df.groupby("problem_id"):
        src = str(g.test.iloc[0])
        ex = split_asserts(src)
        tests[pid] = [expr_test(e) for e in ex] if ex else [dict(name="all tests", script=src)]
    return df[["problem_id", "source_code", "score", "prompt"]], tests

def load_data():
    tests = {}
    if DATA_SOURCE == "demo":
        df, tests = make_demo()
    elif DATA_SOURCE == "hf":
        df = load_falconcode_hf()
    elif DATA_SOURCE == "csv":
        df = load_falconcode_csv()
    elif DATA_SOURCE == "intro_prog":
        df, tests = load_intro_prog()
    else:
        raise ValueError(f"DATA_SOURCE must be 'demo', 'hf', 'csv' or 'intro_prog', got {DATA_SOURCE!r}")
    missing = [c for c in NEEDED if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns {missing}. Found: {list(df.columns)}")
    if TESTS_JSON:
        tests = json.load(open(TESTS_JSON))
    df = df.rename(columns={"source_code": "code"})
    if "type" not in df.columns:
        df["type"] = ""
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    df["problem_id"] = df["problem_id"].astype(str)
    return df, tests

raw, TESTS = load_data()
print(raw.shape, "| columns:", list(raw.columns))

(4394, 5) | columns: ['problem_id', 'code', 'score', 'prompt', 'type']


### Select a purposive sample

In [7]:
def select_sample(df):
    rng = np.random.default_rng(SEED)
    df = df[df["score"] >= 0].dropna(subset=["code", "prompt"]).copy()
    df = df[df["code"].str.count("\n") < MAX_LINES]
    if TYPE_FILTER:
        df = df[df["type"].isin(TYPE_FILTER)]
    df = df.drop_duplicates(["problem_id", "code"])
    n_cor = max(1, round(PER_PROBLEM * FRAC_CORRECT))
    n_inc = PER_PROBLEM - n_cor
    inc = df[df.score < 1].groupby("problem_id").size()
    cor = df[df.score >= 1].groupby("problem_id").size()
    ok = [p for p in inc.index if inc[p] >= n_inc * 2 and cor.get(p, 0) >= n_cor * 2]
    if not ok:
        raise RuntimeError("No problem has enough correct and incorrect submissions; lower PER_PROBLEM.")
    fail = pd.Series({p: inc[p] / (inc[p] + cor[p]) for p in ok}).sort_values()
    pick = np.linspace(0, len(fail) - 1, min(N_PROBLEMS, len(fail))).round().astype(int)
    parts = []
    for p in fail.index[pick]:
        g = df[df.problem_id == p]
        bad = g[g.score < 1].sort_values("score")
        bad = bad.iloc[np.linspace(0, len(bad) - 1, n_inc).round().astype(int)]
        good = g[g.score >= 1]
        good = good.iloc[rng.choice(len(good), n_cor, replace=False)]
        parts += [bad, good]
    s = pd.concat(parts).reset_index(drop=True)
    s["sid"] = [f"s{i:03d}" for i in range(len(s))]
    s["gt_correct"] = (s["score"] >= 1).astype(int)
    return s

sample = select_sample(raw)
print(len(sample), "submissions,", sample.problem_id.nunique(), "problems, correct share:", round(sample.gt_correct.mean(), 2))

50 submissions, 5 problems, correct share: 0.2


### Verification harness

In [8]:
if os.name != "posix":
    print("Windows detected: CPU and memory limits are unavailable (timeout only). Use WSL for the full sandbox.")

BLOCKED_IMPORTS = {"socket", "subprocess", "requests", "urllib", "http", "ftplib", "smtplib", "shutil",
                   "ctypes", "multiprocessing", "pty", "telnetlib", "webbrowser"}
BLOCKED_ATTRS = {"system", "popen", "remove", "unlink", "rmdir", "rmtree", "kill", "fork", "execv", "execl"}

def static_check(code):
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"SyntaxError: {e.msg} (line {e.lineno})", None
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for a in node.names:
                if a.name.split(".")[0] in BLOCKED_IMPORTS:
                    return None, f"imports {a.name}"
        elif isinstance(node, ast.ImportFrom) and node.module and node.module.split(".")[0] in BLOCKED_IMPORTS:
            return None, f"imports {node.module}"
        elif isinstance(node, ast.Attribute) and node.attr in BLOCKED_ATTRS:
            return None, f"calls .{node.attr}"
    return None, None

LAUNCHER = """import sys, runpy
try:
    import resource
    for name, value in (("RLIMIT_CPU", {cpu}), ("RLIMIT_AS", {mem}), ("RLIMIT_FSIZE", {fsz})):
        try:
            resource.setrlimit(getattr(resource, name), (value, value))
        except (ValueError, OSError):
            pass
except ImportError:
    pass
sys.argv = [sys.argv[1]]
runpy.run_path(sys.argv[0], run_name="__main__")
"""

def run_python(code, stdin_text=""):
    with tempfile.TemporaryDirectory() as d:
        f = pathlib.Path(d) / "submission.py"
        f.write_text(code)
        env = {k: os.environ[k] for k in ("PATH", "SYSTEMROOT", "TEMP", "TMP", "COMSPEC") if k in os.environ}
        env.update(PYTHONHASHSEED="0", MPLBACKEND="Agg")
        launcher = LAUNCHER.format(cpu=TIMEOUT_S, mem=MEM_MB * 2**20, fsz=5 * 2**20)
        try:
            r = subprocess.run([sys.executable, "-c", launcher, str(f)], input=stdin_text, capture_output=True, text=True,
                               timeout=TIMEOUT_S + 2, cwd=d, env=env)
            killed = r.returncode in (-24, -9) and not r.stderr.strip()
            return dict(rc=r.returncode, out=r.stdout[-1500:], err="TimeoutExpired" if killed else r.stderr[-1500:], timeout=killed)
        except subprocess.TimeoutExpired:
            return dict(rc=-9, out="", err="TimeoutExpired", timeout=True)

def last_line(s):
    lines = [l for l in s.strip().splitlines() if l.strip()]
    return lines[-1] if lines else ""

def run_expr_test(code, t):
    if "script" in t:
        r = run_python(code + "\n" + t["script"] + "\n")
        return dict(name=t["name"], ok=r["rc"] == 0, expected="", got="", err=last_line(r["err"]) if r["rc"] != 0 else "", raw=True)
    if "left" in t:
        tail = (f"\n__l = ({t['left']})\n__r = ({t['right']})\nprint('@@', __l == __r)\n"
                "print('@@', repr(__l)[:120])\nprint('@@', repr(__r)[:120])\n")
    else:
        tail = f"\nprint('@@', bool({t['expr']}))\n"
    r = run_python(code + "\n" + tail)
    marks = [l[3:] for l in r["out"].splitlines() if l.startswith("@@ ")]
    ok = r["rc"] == 0 and bool(marks) and marks[0] == "True"
    return dict(name=t["name"], ok=ok, expected=marks[2] if len(marks) > 2 else "", got=marks[1] if len(marks) > 1 else "",
                err=last_line(r["err"]) if r["rc"] != 0 else "", raw=True)

def verify(code, score, problem_id):
    syn, blocked = static_check(code)
    sig = dict(syntax=syn, blocked=blocked, run=None, tests=[], score=score)
    if syn or blocked:
        return sig
    tests = TESTS.get(problem_id, [])
    if tests:
        for t in tests:
            if "stdin" not in t:
                sig["tests"].append(run_expr_test(code, t))
                continue
            r = run_python(code, t["stdin"] + "\n")
            got = r["out"].strip()
            ok = r["rc"] == 0 and got == str(t["expected"]).strip()
            sig["tests"].append(dict(name=t["name"], ok=ok, expected=str(t["expected"]).strip(), got=got[:80],
                                     err=last_line(r["err"]) if r["rc"] != 0 else ""))
    else:
        r = run_python(code, "")
        if "EOFError" in r["err"]:
            sig["run"] = "program waits for keyboard input; no test input available"
        elif r["timeout"]:
            sig["run"] = "timed out"
        elif r["rc"] != 0:
            sig["run"] = last_line(r["err"])
        else:
            sig["run"] = "ran without errors"
    return sig

def signal_text(sig):
    first = sig["syntax"] or (f"not run ({sig['blocked']})" if sig["blocked"] else "no syntax errors")
    L = ["Interpreter check: " + first]
    if sig["run"]:
        L.append("Run result: " + sig["run"])
    if sig["tests"]:
        L.append("Unit tests:")
        shown = sig["tests"]
        if len(shown) > 12:
            fails = [t for t in shown if not t["ok"]]
            L.append(f"  {len(shown) - len(fails)} of {len(shown)} tests pass")
            shown = fails[:8]
        for t in shown:
            if t["ok"]:
                L.append(f"  {t['name']}: PASS")
            else:
                exp = (t["expected"], t["got"]) if t.get("raw") else (repr(t["expected"]), repr(t["got"]))
                detail = t["err"] or (f"expected {exp[0]}, got {exp[1]}" if exp[0] or exp[1] else "assertion is false")
                L.append(f"  {t['name']}: FAIL ({detail})")
    if INCLUDE_SCORE_IN_SIGNAL and sig["score"] is not None and not math.isnan(sig["score"]):
        L.append(f"Course autograder score: {sig['score']:.2f} (fraction of the course's test cases passed)")
    return "\n".join(L)

Windows detected: CPU and memory limits are unavailable (timeout only). Use WSL for the full sandbox.


### Database and harness run

In [9]:
con = sqlite3.connect(DB_PATH, check_same_thread=False)
con.executescript("""
CREATE TABLE IF NOT EXISTS submissions(id TEXT PRIMARY KEY, problem_id TEXT, exercise TEXT, code TEXT,
    ground_truth_correct INT, score REAL, signal_text TEXT);
CREATE TABLE IF NOT EXISTS feedback_runs(id INTEGER PRIMARY KEY AUTOINCREMENT, submission_id TEXT, condition TEXT,
    run_k INT, model TEXT, temperature REAL, raw_json TEXT, parsed_ok INT, timestamp TEXT, prompt TEXT,
    UNIQUE(submission_id, condition, run_k));
CREATE TABLE IF NOT EXISTS annotations(id INTEGER PRIMARY KEY AUTOINCREMENT, feedback_run_id INT, coder_id TEXT,
    category TEXT, UNIQUE(feedback_run_id, coder_id));
CREATE TABLE IF NOT EXISTS survey_responses(id INTEGER PRIMARY KEY AUTOINCREMENT, participant_id TEXT, item TEXT, response TEXT);
""")
if "prompt" not in [r[1] for r in con.execute("PRAGMA table_info(feedback_runs)")]:
    con.execute("ALTER TABLE feedback_runs ADD COLUMN prompt TEXT")

old = dict(con.execute("SELECT id, code FROM submissions"))
clash = [r.sid for r in sample.itertuples() if r.sid in old and old[r.sid] != r.code]
if clash:
    raise RuntimeError(f"{DB_PATH} already holds a different sample ({len(clash)} submissions differ). "
                       "Delete that file, or set OUTPUT_DIR to a new folder, then re-run from the Config cell.")

for r in tqdm(sample.itertuples(), total=len(sample)):
    sig = verify(r.code, r.score, r.problem_id)
    con.execute("INSERT OR REPLACE INTO submissions VALUES (?,?,?,?,?,?,?)",
                (r.sid, r.problem_id, r.prompt, r.code, int(r.gt_correct), float(r.score), signal_text(sig)))
con.commit()
print(pd.read_sql("SELECT id, ground_truth_correct gt, substr(signal_text,1,90) signal FROM submissions LIMIT 4", con).to_string())

  0%|          | 0/50 [00:00<?, ?it/s]

     id  gt                                      signal
0  s000   0  Interpreter check: not run (calls .remove)
1  s001   0  Interpreter check: not run (calls .remove)
2  s002   0  Interpreter check: not run (calls .remove)
3  s003   0  Interpreter check: not run (calls .remove)


### Prompts, LLM client, parsing

In [1]:
SYSTEM = ("You are a programming tutor for an introductory Python course. Judge whether the student's code solves the "
          "assignment and give short hints. Never reveal the full corrected solution and never write corrected code.")

SCHEMA = ('Return ONLY a JSON object, no other text:\n'
          '{"is_correct": true or false,\n'
          ' "hints": [{"line": <int>, "code_line": "<exact text of that line>", "explanation": "<short hint, no corrected code>"}]}\n'
          'If the code is correct return is_correct true and an empty hints list. Give at most 2 hints.')

def numbered(code):
    return "\n".join(f"{i+1}: {l}" for i, l in enumerate(code.splitlines()))

def build_prompt(exercise, code, condition, sig_text):
    p = f"Assignment:\n{exercise}\n\nStudent code (line numbers shown):\n{numbered(code)}\n"
    if condition == "B":
        p += ("\nVerification results for this code:\n" + sig_text +
              "\n\nBase your hints on what these results show. Do not claim problems the results do not support.\n")
    return p + "\n" + SCHEMA

def _mock(prompt):
    rng = random.Random(int(hashlib.md5(prompt.encode()).hexdigest(), 16))
    lines = re.findall(r"^(\d+): (.*)$", prompt, re.M)
    if "Verification results" in prompt:
        correct = not (("FAIL" in prompt) or ("Error" in prompt) or ("score:" in prompt and "score: 1.00" not in prompt))
    else:
        correct = rng.random() < 0.45
    if correct:
        return json.dumps(dict(is_correct=True, hints=[]))
    ln, txt = lines[rng.randrange(len(lines))] if lines else ("1", "")
    return json.dumps(dict(is_correct=False, hints=[dict(line=int(ln), code_line=txt, explanation="Check the logic on this line.")]))

_clients = {}

def call_llm(prompt):
    if LLM_PROVIDER == "mock":
        return _mock(prompt)
    if LLM_PROVIDER == "openai":
        if "openai" not in _clients:
            ensure("openai")
            from openai import OpenAI
            _clients["openai"] = OpenAI(api_key=get_secret("OPENAI_API_KEY"))
        r = _clients["openai"].chat.completions.create(
            model=OPENAI_MODEL, temperature=TEMPERATURE, response_format={"type": "json_object"},
            messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    if LLM_PROVIDER == "anthropic":
        if "anthropic" not in _clients:
            ensure("anthropic")
            import anthropic
            _clients["anthropic"] = anthropic.Anthropic(api_key=get_secret("ANTHROPIC_API_KEY"))
        r = _clients["anthropic"].messages.create(
            model=ANTHROPIC_MODEL, max_tokens=700, temperature=TEMPERATURE, system=SYSTEM,
            messages=[{"role": "user", "content": prompt}])
        return r.content[0].text
    raise ValueError(f"LLM_PROVIDER must be 'mock', 'openai' or 'anthropic', got {LLM_PROVIDER!r}")

def parse_hint(text):
    m = re.search(r"\{.*\}", text or "", re.S)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    if not isinstance(obj.get("is_correct"), bool) or not isinstance(obj.get("hints", []), list):
        return None
    hints = [dict(line=h.get("line"), code_line=str(h.get("code_line", "")), explanation=str(h.get("explanation", "")))
             for h in obj.get("hints", [])[:2] if isinstance(h, dict)]
    return dict(is_correct=obj["is_correct"], hints=hints)

def render_hint(p):
    head = "Verdict: code is " + ("CORRECT" if p["is_correct"] else "INCORRECT")
    return head + "".join(f"\n  Line {h['line']}: `{h['code_line']}` - {h['explanation']}" for h in p["hints"])

def one_call(prompt):
    last = None
    for attempt in range(4):
        try:
            text = call_llm(prompt)
            parsed = parse_hint(text)
            if parsed is None:
                text = call_llm(prompt)
                parsed = parse_hint(text)
            return text, parsed
        except Exception as e:
            last = e
            time.sleep(2 ** attempt)
    raise last

### Run both conditions

In [11]:
MODEL_NAME = {"mock": "mock", "openai": OPENAI_MODEL, "anthropic": ANTHROPIC_MODEL}[LLM_PROVIDER]

def run_batch():
    done = {(a, b, c) for a, b, c in con.execute("SELECT submission_id, condition, run_k FROM feedback_runs")}
    subs = {r[0]: r for r in con.execute("SELECT id, exercise, code, signal_text FROM submissions")}
    jobs = [(sid, c, k) for sid in subs for c in ("A", "B") for k in range(K_SAMPLES) if (sid, c, k) not in done]
    print(len(jobs), "calls to make,", len(done), "already done")

    def work(job):
        sid, c, k = job
        _, ex, code, sig = subs[sid]
        prompt = build_prompt(ex, code, c, sig)
        text, parsed = one_call(prompt)
        return sid, c, k, text, parsed, prompt

    failed = 0
    with ThreadPoolExecutor(MAX_WORKERS) as pool:
        futs = {pool.submit(work, j): j for j in jobs}
        for f in tqdm(as_completed(futs), total=len(futs)):
            try:
                sid, c, k, text, parsed, prompt = f.result()
            except Exception as e:
                failed += 1
                print("failed:", futs[f], repr(e)[:120])
                continue
            con.execute("INSERT OR REPLACE INTO feedback_runs(submission_id,condition,run_k,model,temperature,raw_json,parsed_ok,timestamp,prompt) "
                        "VALUES (?,?,?,?,?,?,?,?,?)",
                        (sid, c, k, MODEL_NAME, TEMPERATURE, json.dumps(parsed) if parsed else text, int(parsed is not None),
                         time.strftime("%Y-%m-%d %H:%M:%S"), prompt))
            con.commit()
    print("failed calls (re-run this cell to retry):", failed)

run_batch()
print(pd.read_sql("SELECT condition, COUNT(*) n, SUM(parsed_ok) parsed_ok FROM feedback_runs GROUP BY condition", con))

300 calls to make, 0 already done


  0%|          | 0/300 [00:00<?, ?it/s]

failed calls (re-run this cell to retry): 0
  condition    n  parsed_ok
0         A  150        150
1         B  150        150


### Automatic smoke check

In [12]:
from statsmodels.stats.contingency_tables import mcnemar

def majority(bools):
    return int(sum(bools) * 2 > len(bools))

def paired_mcnemar(wide, a="A", b="B"):
    w = wide.dropna(subset=[a, b])
    t = [[int(((w[a] == 1) & (w[b] == 1)).sum()), int(((w[a] == 1) & (w[b] == 0)).sum())],
         [int(((w[a] == 0) & (w[b] == 1)).sum()), int(((w[a] == 0) & (w[b] == 0)).sum())]]
    return t, mcnemar(t, exact=True).pvalue, w[a].mean(), w[b].mean(), len(w)

runs = pd.read_sql("""SELECT r.id run_id, r.submission_id sid, r.condition, r.run_k, r.raw_json, r.parsed_ok,
                             s.ground_truth_correct gt_correct, s.problem_id
                      FROM feedback_runs r JOIN submissions s ON s.id = r.submission_id""", con)
runs = runs[runs.parsed_ok == 1].copy()
runs["parsed"] = runs.raw_json.apply(json.loads)
runs["verdict_ok"] = [int(p["is_correct"] == bool(g)) for p, g in zip(runs.parsed, runs.gt_correct)]

wide = runs.groupby(["sid", "condition"]).verdict_ok.apply(lambda s: majority(list(s))).unstack()
t, p, accA, accB, n = paired_mcnemar(wide)
print(f"verdict accuracy  A={accA:.2f}  B={accB:.2f}  n={n}  McNemar exact p={p:.4f}")
print("paired table rows=A(ok,bad) cols=B(ok,bad):", t)
print("Plumbing check only: Condition B can see the autograder score. Use the human-coded result below.")

verdict accuracy  A=0.86  B=0.96  n=50  McNemar exact p=0.0625
paired table rows=A(ok,bad) cols=B(ok,bad): [[43, 0], [5, 2]]
Plumbing check only: Condition B can see the autograder score. Use the human-coded result below.


### Export annotation sheet

In [13]:
CATS = {"AC": "Accurate and Complete: right line and right explanation, or correctly says the code is correct with no hints",
        "ALO": "Accurate Line Only: right line, wrong or misleading explanation",
        "AIO": "Accurate Issue Only: describes a real issue but points to the wrong line",
        "FP": "False Positive: flags an issue that does not exist, including hints on correct code",
        "MS": "Misleading Suggestion: a real error exists but line and explanation are both wrong. "
              "Also used when the model says the code is correct but it is not."}

allruns = pd.read_sql("""SELECT r.id run_id, r.condition, r.raw_json, r.parsed_ok, s.exercise, s.code, s.score, s.ground_truth_correct
                         FROM feedback_runs r JOIN submissions s ON s.id = r.submission_id""", con)
allruns = allruns[allruns.parsed_ok == 1].sample(frac=1, random_state=SEED).reset_index(drop=True)
sheet = pd.DataFrame({
    "run_id": allruns.run_id,
    "assignment": allruns.exercise,
    "student_code_numbered": allruns.code.apply(numbered),
    "course_autograder_score": allruns.score,
    "llm_hint": allruns.raw_json.apply(lambda s: render_hint(json.loads(s))),
    "coder2_subset": (np.random.default_rng(SEED).random(len(allruns)) < CODER2_FRACTION).astype(int),
    "category_coder1": "", "category_coder2": "", "final_category": ""})
sheet_path = os.path.join(WORKDIR, "annotation_sheet.csv")
sheet.to_csv(sheet_path, index=False)
pd.Series(CATS).to_csv(os.path.join(WORKDIR, "category_guide.csv"), header=["definition"])
print(f"wrote {sheet_path} ({len(sheet)} rows, {int(sheet.coder2_subset.sum())} double-coded) and category_guide.csv")
print("Coder 1 labels every row. Coder 2 labels rows with coder2_subset=1. Agreed labels for disagreements go in final_category.")

wrote c:\kaggle\working\annotation_sheet.csv (300 rows, 85 double-coded) and category_guide.csv
Coder 1 labels every row. Coder 2 labels rows with coder2_subset=1. Agreed labels for disagreements go in final_category.


### Simulated labels (demo only)

In [14]:
if DATA_SOURCE == "demo" and ANNOT_CSV is None:
    rng = np.random.default_rng(0)
    sim = sheet.copy()
    truth = allruns.assign(p=allruns.raw_json.apply(json.loads))
    base = [("AC" if p["is_correct"] == bool(g) else "FP") for p, g in zip(truth.p, truth.ground_truth_correct)]
    flip = rng.random(len(sim)) < 0.08
    sim["category_coder1"] = base
    sim["category_coder2"] = [("MS" if f else b) if m == 1 else "" for b, f, m in zip(base, flip, sim.coder2_subset)]
    ANNOT_CSV = os.path.join(WORKDIR, "annotation_sheet_SIMULATED.csv")
    sim.to_csv(ANNOT_CSV, index=False)
    print("demo labels written to", ANNOT_CSV)

### Kappa, majority vote, McNemar

In [22]:
from sklearn.metrics import cohen_kappa_score

if ANNOT_CSV:
    lab = pd.read_csv(ANNOT_CSV, dtype=str).fillna("")
    lab["run_id"] = lab["run_id"].astype(int)
    lab["category"] = np.where(lab.final_category != "", lab.final_category, lab.category_coder1)
    bad_rows = lab.loc[~lab.category.isin(CATS), "run_id"].tolist()
    assert not bad_rows, f"unlabeled or invalid rows (first 10): {bad_rows[:10]}"
    for r in lab.itertuples():
        for coder, cat in (("final", r.category), ("coder1", r.category_coder1), ("coder2", r.category_coder2)):
            if cat:
                con.execute("INSERT OR REPLACE INTO annotations(feedback_run_id, coder_id, category) VALUES (?,?,?)", (r.run_id, coder, cat))
    con.commit()

    both = lab[(lab.category_coder1 != "") & (lab.category_coder2 != "")]
    kappa5 = cohen_kappa_score(both.category_coder1, both.category_coder2)
    kappa2 = cohen_kappa_score(both.category_coder1 == "AC", both.category_coder2 == "AC")
    print(f"agreement on {len(both)} double-coded hints: kappa(5 categories)={kappa5:.2f}, kappa(AC vs not)={kappa2:.2f}")
    print("gate >= 0.70:", "PASS" if kappa5 >= 0.7 else "FAIL: discuss disagreements, fill final_category, re-run")

    meta = pd.read_sql("SELECT id run_id, submission_id sid, condition, run_k FROM feedback_runs", con)
    lab = lab.merge(meta, on="run_id")

    def agg(g):
        cats = list(g.category)
        return pd.Series(dict(ac=majority([c == "AC" for c in cats]),
                              category=pd.Series(cats).value_counts().index[0],
                              three_way_split=int(len(cats) >= 3 and len(set(cats)) == len(cats))))

    item = lab.groupby(["sid", "condition"]).apply(agg).reset_index()
    print("three-way splits for manual review:", int(item.three_way_split.sum()))
    wide = item.pivot(index="sid", columns="condition", values="ac")
    t, p, accA, accB, n = paired_mcnemar(wide)
    print(f"\nhint accuracy (AC): A={accA:.2f}  B={accB:.2f}  n={n}")
    print("paired table rows=A(ok,bad) cols=B(ok,bad):", t)
    print(f"McNemar exact p = {p:.4f}  (A-only correct={t[0][1]}, B-only correct={t[1][0]})")

    dist = lab.groupby(["condition", "category"]).size().unstack(fill_value=0).reindex(columns=list(CATS), fill_value=0)
    dist_pct = dist.div(dist.sum(axis=1), axis=0).round(3)
    print("\ncategory distribution:\n", dist_pct)

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ax = dist_pct.T.plot(kind="bar", figsize=(7, 4), color=["#8a8a8a", "#1E4D2B"], rot=0)
    ax.set_ylabel("share of hints")
    ax.set_title("Error-coding categories by condition (A = code only, B = + verification)")
    plt.tight_layout()
    plt.savefig(os.path.join(WORKDIR, "category_distribution.png"), dpi=150)
    try:
        from IPython.display import display
        display(ax.figure)
    except Exception:
        pass
    json.dump(dict(kappa5=float(kappa5), kappa_binary=float(kappa2), accA=float(accA), accB=float(accB), n=int(n),
                   mcnemar_p=float(p), table=[[int(v) for v in row] for row in t],
                   n_double=int(len(both)), n_hints=int(len(lab)), three_way=int(item.three_way_split.sum()),
                   dist={str(c): {str(k): float(v) for k, v in row.items()} for c, row in dist_pct.T.iterrows()}),
              open(os.path.join(WORKDIR, "results.json"), "w"), indent=2)
else:
    print("Set ANNOT_CSV to the filled-in annotation sheet and re-run this cell.")

Set ANNOT_CSV to the filled-in annotation sheet and re-run this cell.
